# 对比检字表

In [ ]:
import pandas as pd
from typing import List, Dict, Tuple
import re

def compare_and_generate_report(ocr_table: List[Dict], corrected_table: List[Dict]) -> pd.DataFrame:
    """
    比较OCR表格和人工校验表格，生成差异统计报告
    
    参数:
        ocr_table: OCR识别结果表格，格式为[{"笔画数": "...", "字": "..."}, ...]
        corrected_table: 人工校验表格，格式同ocr_table
        
    返回:
        包含差异统计的DataFrame
    """
    report_rows = []
    
    # 创建一个字典，方便按笔画数查找
    ocr_dict = {row["笔画数"]: row["字"] for row in ocr_table}
    corrected_dict = {row["笔画数"]: row["字"] for row in corrected_table}
    
    for stroke_count in ocr_dict:
        if stroke_count not in corrected_dict:
            continue  # 如果笔画数不在校验表中，跳过
            
        ocr_chars = ocr_dict[stroke_count]
        corrected_chars = corrected_dict[stroke_count]
        
        # 逐个字符比较
        min_len = min(len(ocr_chars), len(corrected_chars))
        
        for i in range(min_len):
            if ocr_chars[i] != corrected_chars[i]:
                # 构建来源描述
                source = f"检字表{stroke_count}第{i+1}字"
                
                # 添加行到报告
                report_rows.append({
                    "OCR结果": ocr_chars[i],
                    "来源": source,
                    "错误类型": "识别错误",
                    "清洗后": corrected_chars[i]
                })
        
        # 处理长度不匹配的情况（如果有）
        if len(ocr_chars) != len(corrected_chars):
            # 这里可能需要根据具体需求调整处理逻辑
            # 当前示例中没有这种情况，所以暂时不处理
            pass
    
    return pd.DataFrame(report_rows)

def parse_markdown_table(markdown_text: str) -> List[Dict]:
    """
    解析markdown表格文本为字典列表
    
    参数:
        markdown_text: markdown格式的表格文本
        
    返回:
        解析后的数据列表
    """
    lines = markdown_text.strip().split('\n')
    
    # 提取表头
    headers = []
    data = []
    
    for i, line in enumerate(lines):
        line = line.strip()
        
        # 跳过空行
        if not line:
            continue
            
        # 移除表格分隔符|
        cells = [cell.strip() for cell in line.split('|') if cell.strip()]
        
        # 第一行是表头
        if i == 0:
            headers = cells
        # 第二行是分隔线，跳过
        elif i == 1:
            continue
        else:
            # 数据行
            if len(cells) == len(headers):
                row = {}
                for j in range(len(headers)):
                    row[headers[j]] = cells[j]
                data.append(row)
    
    return data

def generate_markdown_report(df: pd.DataFrame) -> str:
    """
    将DataFrame转换为markdown表格格式
    
    参数:
        df: 包含统计数据的DataFrame
        
    返回:
        markdown格式的表格字符串
    """
    if df.empty:
        return "没有发现差异"
    
    markdown_lines = ["| OCR结果 | 来源 | 错误类型 | 清洗后 |"]
    markdown_lines.append("| :--- | :--- | :--- | :--- |")
    
    for _, row in df.iterrows():
        markdown_lines.append(f"| {row['OCR结果']} | {row['来源']} | {row['错误类型']} | {row['清洗后']} |")
    
    return "\n".join(markdown_lines)

# 示例数据
ocr_markdown = """| 笔画数 | 字 |
| :--- | :--- |
| 二画  | 丁七八八九 |
| 三画  | 三土下大万上口山门小飞叉马子 |
| 四画  | 切云天井木瓦五牙厅止日内仓分手牛乌勾丹月计方火斗心双水|
| 五画  |打平正布石龙出由四外卯令生白瓜汉立永对  |
| 六画  | 地耳夹列托压当曲曳竹伏仰华行合杂名齐交次安讹寻阳阶如红欢 |
| 七画  |  坛材杚苇花走两束扶批拒抄折抢护把连吴足里串帐身佛彻余肘角龟条间沥沙补附鸡纴|
| 八画  | 青坯取枨板直抹抽抱转软卧明昂罗垂侧侏乳金斧股底卷单泥宝定实衬驼线细承 |
| 九画  | 项城栋栌相柎柱胡草砖斫挟挑耍面点背贴虹虾虻钩重促顺须盆亭将举屋垒结绞 |
| 十画  |  素栔桩栱枤格荻起破套剔柴铁笏透脊狼鸱鸾流调阁展难|
| 十一画  | 琉梢桯棂梭壶副营黄曹厢捧排虚堂常偷斜彩象旋望廊减混粗断剪兽着窑随隐骑续绰 |
| 十二画  | 替琴棵榁𬃊棚散惹葱趄欹厦雁颊搭插辋赑铺锭鹅筒牌竣敦阑普隔缘 |
| 十三画  |  榔椽鹊靴鼓蒜搕搏辐暗睒跳罨蜀照腰锯解障殿盏嫔叠缝缠|
| 十四画  |墙檼槛榥榻槏截劄算箫蝉裹褊遮慢滴  |
| 十五画  | 增横槽榑碾撮撺幡踏影颛镇篆熟额撩 |
| 十六画  | 檂燕搢螭雕磨瘿壁缴 |
| 十七画  | 墬檐藏螳镫簇爵 |
| 十八画  | 鳌覆鹰鹰 |
| 十九画  | 藻蹲瓣 |
| 二十一画  | 露 |
| 二十四画  | 褥 |
| 二十五画  | 镕 |"""

corrected_markdown = """| 笔画数 | 字 |
| :--- | :--- |
| 二画  | 丁七八入九 |
| 三画  | 三土下大万上口山门小飞叉马子 |
| 四画  | 切云天井木瓦五牙厅止日内仓分手牛乌勾丹月计方火斗心双水|
| 五画  |打平正布石龙出由四外卯令生白瓜汉立永对  |
| 六画  | 地耳夹列托压当曲曳竹伏仰华行合杂名齐交次安讹寻阳阶如红欢 |
| 七画  |  坛材杚苇花走两束扶批拒抄折抢护把连吴足里串帐身佛彻余肘角龟条间沥沙补附鸡纴|
| 八画  | 青坯取枨板直抹抽抱转软卧明昂罗垂侧侏乳金斧股底卷单泥宝定实衬驼线细承 |
| 九画  | 项城栋栌相柎柱胡草砖斫挟挑耍面点背贴虹虾虻钩重促顺须盆亭将举屋垒结绞 |
| 十画  |  素栔桩栱枤格荻起破套剔柴铁笏透脊狼鸱鸳流调？展难|
| 十一画  | 琉梢桯棂梭壶副营黄曹厢捧排虚堂常偷斜彩象旋望廊减混粗断剪兽着窑随隐骑续绰 |
| 十二画  | 替琴棵㭼𬃊棚散惹葱趄欹厦雁颊搭插辋赑铺？鹅筒牌竣敦阑普隔缘 |
| 十三画  |  楅椽鹊靴鼓蒜搕搏辐暗睒跳罨蜀照腰锯解障殿盝嫔叠缝缠|
| 十四画  |墙？槛榥榻槏截劄算箫蝉裹褊遮慢滴  |
| 十五画  | 增横槽榑碾撮撺幡踏？颛镇篆熟额撩 |
| 十六画  | 橑燕擗螭雕磨瘿壁缴 |
| 十七画  | 壕檐藏螳镫簇爵 |
| 十八画  | 鳌覆藕鹰 |
| 十九画  | 藻蹲瓣 |
| 二十一画  | 露 |
| 二十四画  | 襻 |
| 二十五画  | ? |"""

def main():
    """主函数，执行比较并生成报告"""
    # 解析markdown表格
    ocr_data = parse_markdown_table(ocr_markdown)
    corrected_data = parse_markdown_table(corrected_markdown)
    
    # print("OCR表格数据:")
    # for row in ocr_data:
    #     # print(f"笔画数: {row['笔画数']}, 字: {row['字']}")
    # print()
    
    # print("人工校验表格数据:")
    # for row in corrected_data:
    #     # print(f"笔画数: {row['笔画数']}, 字: {row['字']}")
    # print()
    
    # 比较并生成报告
    report_df = compare_and_generate_report(ocr_data, corrected_data)
    
    print("差异统计报告:")
    print(generate_markdown_report(report_df))
    
    # # 输出详细的差异分析
    # print("\n详细差异分析:")
    # for _, row in report_df.iterrows():
    #     print(f"{row['来源']}: '{row['OCR结果']}' -> '{row['清洗后']}'")
    
    return report_df

if __name__ == "__main__":
    # 运行主函数
    report = main()
    
    # 如果你需要将报告保存为文件
    # report.to_csv("字符校正统计报告.csv", index=False, encoding='utf-8-sig')
    # print("\n报告已保存为 '字符校正统计报告.csv'")

差异统计报告:
| OCR结果 | 来源 | 错误类型 | 清洗后 |
| :--- | :--- | :--- | :--- |
| 八 | 检字表二画第4字 | 识别错误 | 入 |
| 鸾 | 检字表十画第19字 | 识别错误 | 鸳 |
| 阁 | 检字表十画第22字 | 识别错误 | ？ |
| 榁 | 检字表十二画第4字 | 识别错误 | 㭼 |
| 锭 | 检字表十二画第20字 | 识别错误 | ？ |
| 榔 | 检字表十三画第1字 | 识别错误 | 楅 |
| 盏 | 检字表十三画第21字 | 识别错误 | 盝 |
| 檼 | 检字表十四画第2字 | 识别错误 | ？ |
| 影 | 检字表十五画第10字 | 识别错误 | ？ |
| 檂 | 检字表十六画第1字 | 识别错误 | 橑 |
| 搢 | 检字表十六画第3字 | 识别错误 | 擗 |
| 墬 | 检字表十七画第1字 | 识别错误 | 壕 |
| 鹰 | 检字表十八画第3字 | 识别错误 | 藕 |
| 褥 | 检字表二十四画第1字 | 识别错误 | 襻 |
| 镕 | 检字表二十五画第1字 | 识别错误 | ？ |


# 对比整个文件，输出修正清单

## 步骤1：解析Markdown表格

In [ ]:
import re
from typing import List, Dict, Tuple, Optional
import json

def parse_markdown_tables(file_path: str) -> Dict[str, Dict]:
    """
    解析Markdown文件，提取所有一级标题、二级标题和对应的表格数据
    
    返回结构：
    {
        "一级标题1": {
            "二级标题1": {
                "headers": ["列1", "列2", ...],
                "rows": [["行1列1", "行1列2", ...], ...]
            },
            ...
        },
        ...
    }
    """
    
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # 初始化结果字典
    tables = {}
    
    # 使用正则表达式找到所有一级标题、二级标题和表格
    lines = content.split('\n')
    
    current_level1 = ""
    current_level2 = ""
    in_table = False
    table_headers = []
    table_rows = []
    reading_headers = False
    reading_data = False
    
    for i, line in enumerate(lines):
        line = line.strip()
        
        # 匹配一级标题 (格式: # 标题)
        level1_match = re.match(r'^#\s+(.+)$', line)
        if level1_match and not line.startswith('##'):
            # 保存之前的表格（如果有）
            if in_table and current_level1 and current_level2 and table_rows:
                if current_level1 not in tables:
                    tables[current_level1] = {}
                tables[current_level1][current_level2] = {
                    "headers": table_headers,
                    "rows": table_rows
                }
            
            # 重置状态
            current_level1 = level1_match.group(1)
            current_level2 = ""
            in_table = False
            table_headers = []
            table_rows = []
            continue
        
        # 匹配二级标题 (格式: ## 标题)
        level2_match = re.match(r'^##\s+(.+)$', line)
        if level2_match:
            # 保存之前的表格（如果有）
            if in_table and current_level1 and current_level2 and table_rows:
                if current_level1 not in tables:
                    tables[current_level1] = {}
                tables[current_level1][current_level2] = {
                    "headers": table_headers,
                    "rows": table_rows
                }
            
            # 重置状态
            current_level2 = level2_match.group(1)
            in_table = False
            table_headers = []
            table_rows = []
            continue
        
        # 匹配表格开始（包含|和-的行，通常是表头分隔线）
        if re.match(r'^\|?\s*:?-+:?\s*\|', line) or re.match(r'^\|?\s*-+\s*\|', line):
            # 这是一个表头分隔线，表示表格开始
            # 获取表头行（上一行）
            if i > 0 and not in_table:
                header_line = lines[i-1].strip()
                if '|' in header_line:
                    # 提取表头
                    table_headers = extract_table_cells(header_line)
                    in_table = True
                    reading_data = True
            continue
        
        # 匹配表格数据行
        if in_table and reading_data and '|' in line and not re.match(r'^\|?\s*:?-+:?\s*\|', line):
            # 确保不是表头分隔线
            if not re.match(r'^\|?\s*-+\s*\|', line):
                row_data = extract_table_cells(line)
                if len(row_data) == len(table_headers):
                    table_rows.append(row_data)
                elif len(row_data) > 0:
                    # 如果列数不匹配，尝试调整
                    print(f"警告: 行 {i+1} 列数不匹配: 预期 {len(table_headers)} 列, 实际 {len(row_data)} 列")
                    # 尝试填充或截断
                    if len(row_data) > len(table_headers):
                        row_data = row_data[:len(table_headers)]
                    else:
                        row_data.extend([''] * (len(table_headers) - len(row_data)))
                    table_rows.append(row_data)
    
    # 保存最后一个表格
    if in_table and current_level1 and current_level2 and table_rows:
        if current_level1 not in tables:
            tables[current_level1] = {}
        tables[current_level1][current_level2] = {
            "headers": table_headers,
            "rows": table_rows
        }
    
    return tables

def extract_table_cells(line: str) -> List[str]:
    """
    从Markdown表格行中提取单元格内容
    
    参数:
        line: 表格行字符串
        
    返回:
        单元格内容列表
    """
    # 移除行首尾的管道符号和空格
    line = line.strip()
    if line.startswith('|'):
        line = line[1:]
    if line.endswith('|'):
        line = line[:-1]
    
    # 分割单元格
    cells = []
    current_cell = ""
    in_code = False  # 用于处理单元格内的代码标记
    
    i = 0
    while i < len(line):
        char = line[i]
        
        if char == '|' and not in_code:
            cells.append(current_cell.strip())
            current_cell = ""
        elif char == '`':
            in_code = not in_code
            current_cell += char
        else:
            current_cell += char
        i += 1
    
    # 添加最后一个单元格
    if current_cell:
        cells.append(current_cell.strip())
    
    return cells

def parse_markdown_tables_alternative(file_path: str) -> Dict[str, Dict]:
    """
    备选解析方案：使用更简单的方法解析Markdown表格
    
    这个方法直接查找二级标题，然后查找紧随其后的表格
    """
    
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # 按行分割
    lines = content.split('\n')
    
    tables = {}
    current_level1 = ""
    current_level2 = ""
    
    i = 0
    while i < len(lines):
        line = lines[i].strip()
        
        # 匹配一级标题
        if line.startswith('# ') and not line.startswith('## '):
            current_level1 = line[2:].strip()
            if current_level1 not in tables:
                tables[current_level1] = {}
        
        # 匹配二级标题
        elif line.startswith('## '):
            current_level2 = line[3:].strip()
        
        # 查找表格
        elif line.startswith('|'):
            # 找到表格开始
            # 获取表头行
            header_line = line
            
            # 检查下一行是否是表头分隔线
            if i + 1 < len(lines) and '|' in lines[i+1] and ('---' in lines[i+1] or ':' in lines[i+1]):
                # 提取表头
                headers = extract_table_cells_simple(header_line)
                
                # 跳过表头分隔线
                i += 1
                
                # 收集数据行
                rows = []
                i += 1
                
                while i < len(lines) and lines[i].strip().startswith('|'):
                    data_line = lines[i].strip()
                    # 跳过空行或分隔线
                    if not data_line or '---' in data_line:
                        i += 1
                        continue
                    
                    row_data = extract_table_cells_simple(data_line)
                    if row_data:
                        rows.append(row_data)
                    i += 1
                
                # 保存表格数据
                if current_level1 and current_level2 and headers and rows:
                    if current_level1 not in tables:
                        tables[current_level1] = {}
                    tables[current_level1][current_level2] = {
                        "headers": headers,
                        "rows": rows
                    }
                
                # 继续处理下一行
                continue
        
        i += 1
    
    return tables

def extract_table_cells_simple(line: str) -> List[str]:
    """
    简单提取表格单元格
    
    参数:
        line: 表格行字符串
        
    返回:
        单元格内容列表
    """
    # 移除行首尾的管道符号
    line = line.strip()
    if line.startswith('|'):
        line = line[1:]
    if line.endswith('|'):
        line = line[:-1]
    
    # 简单的分割
    cells = [cell.strip() for cell in line.split('|')]
    return cells

def display_table_summary(tables: Dict[str, Dict]) -> None:
    """显示表格摘要信息"""
    print("=" * 60)
    print("表格解析摘要")
    print("=" * 60)
    
    total_tables = 0
    total_rows = 0
    
    for level1, level2_dict in tables.items():
        print(f"\n一级标题: {level1}")
        
        if not level2_dict:
            print(f"  (无表格)")
            continue
            
        for level2, table_data in level2_dict.items():
            total_tables += 1
            headers = table_data["headers"]
            row_count = len(table_data["rows"])
            total_rows += row_count
            
            print(f"  ├─ 二级标题: {level2}")
            print(f"  │  ├─ 表头: {headers}")
            print(f"  │  ├─ 行数: {row_count}")
            print(f"  │  └─ 列数: {len(headers)}")
    
    print(f"\n总计: {total_tables} 个表格, {total_rows} 行数据")
    print("=" * 60)

# 测试解析功能
if __name__ == "__main__":
    # 使用您的示例数据创建测试文件
    test_content = """# 检字表
## 检字
| 笔画数 | 字 |
| :--- | :--- |
| 二画  | 丁七八八九 |
| 三画  | 三土下大万上口山门小飞叉马子 |
| 四画  | 切云天井木瓦五牙厅止日内仓分手牛乌勾丹月计方火斗心双水|
| 十八画  | 鳌覆鹰鹰 |
| 十九画  | 藻蹲瓣 |
| 二十一画  | 露 |
| 二十四画  | 褥 |
| 二十五画  | 镕 |

## 替换字表
| 现代汉语用字 | 《法式》用字 |
| :--- | :--- |
| 斗 | 闟、枓 |
| 只 | 隻 |
| 曳 | 拽 |
| 着 | 著 |
| 苘 | 蒽 |
| 筒 | 瓿 |
| 棋 | 棊 |
| 遍 | 偏 |
| 靴 | 鞭 |
| 暗 | 闇 |
| 鳌 | ? |

# 术语解释表
## 二画
| 术语 | 首次出现卷数 | 解释 |
| :--- | :--- | :--- |
| 丁栿 | (4) | 用于山面的纵向梁栿。 |
| 丁华抹颏栱 | (4) | 脊部叉手上角内，蜀柱上横向出耍头之栱。 |
| 丁头栱 | (4) | 只有一卷头的半截栱。 |

## 三画
| 术语 | 首次出现卷数 | 解释 |
| :--- | :--- | :--- |
| 三晕棱间装 | (14) | 青绿叠晕棱间装的一种, 在两晕棱间装的身内再重复一层外棱之颜色。 |
| 三晕带棱 | (14) | 青绿叠晕棱间装的一种, 外棱与身内用青绿相间叠晕, 而中间用红色叠晕。 |
| 间装 | 无 | (暂无具体解释) |
| 测试术语 | (1) | 这是一个测试术语，用于验证解析功能。 |
"""
    
    with open("test_markdown.md", "w", encoding="utf-8") as f:
        f.write(test_content)
    
    # print("使用第一种方法解析:")
    # print("-" * 60)
    # tables1 = parse_markdown_tables("test_markdown.md")
    # display_table_summary(tables1)
    
    print("\n\n解析markdownn表格")
    ModifyTables = parse_markdown_tables_alternative(r"knowledgeBase\cleaned_data\《营造法式》解读 第2版术语库_Cleaned.md")
    display_table_summary(ModifyTables)

    OCRtables = parse_markdown_tables_alternative(r"knowledgeBase\pdfParsed_output\《营造法式》解读 第2版术语库_Pages[1-25]_格式修正.md")
    display_table_summary(OCRtables)
    
    # 保存解析结果
    # with open("parsed_tables.json", "w", encoding="utf-8") as f:
    #     json.dump(tables2, f, ensure_ascii=False, indent=2)




解析markdownn表格
表格解析摘要

一级标题: 检字表
  ├─ 二级标题: 检字
  │  ├─ 表头: ['笔画数', '字']
  │  ├─ 行数: 21
  │  └─ 列数: 2
  ├─ 二级标题: 替换字
  │  ├─ 表头: ['现代汉语用字', '《法式》用字']
  │  ├─ 行数: 21
  │  └─ 列数: 2

一级标题: 术语解释表
  ├─ 二级标题: 二画
  │  ├─ 表头: ['术语', '首次出现卷数', '解释']
  │  ├─ 行数: 10
  │  └─ 列数: 3
  ├─ 二级标题: 三画
  │  ├─ 表头: ['术语', '首次出现卷数', '解释']
  │  ├─ 行数: 45
  │  └─ 列数: 3
  ├─ 二级标题: 四画
  │  ├─ 表头: ['术语', '首次出现卷数', '解释']
  │  ├─ 行数: 66
  │  └─ 列数: 3
  ├─ 二级标题: 五 画
  │  ├─ 表头: ['术语', '首次出现卷数', '解释']
  │  ├─ 行数: 53
  │  └─ 列数: 3
  ├─ 二级标题: 六画
  │  ├─ 表头: ['术语', '首次出现卷数', '解释']
  │  ├─ 行数: 65
  │  └─ 列数: 3
  ├─ 二级标题: 七画
  │  ├─ 表头: ['术语', '首次出现卷数', '解释']
  │  ├─ 行数: 65
  │  └─ 列数: 3
  ├─ 二级标题: 八画
  │  ├─ 表头: ['术语', '首次出现卷数', '解释']
  │  ├─ 行数: 87
  │  └─ 列数: 3
  ├─ 二级标题: 九画
  │  ├─ 表头: ['术语', '首次出现卷数', '解释']
  │  ├─ 行数: 57
  │  └─ 列数: 3
  ├─ 二级标题: 十画
  │  ├─ 表头: ['术语', '首次出现卷数', '解释']
  │  ├─ 行数: 42
  │  └─ 列数: 3
  ├─ 二级标题: 十一画
  │  ├─ 表头: ['术语', '首次出现卷数', '解释']
  │  ├─ 行数: 54
  │  └─ 列数: 3
  ├─ 二级标题: 十二画
  │  ├─ 表头:

## 步骤2：为表格添加序号列

In [14]:
def add_serial_number_to_tables(tables: Dict[str, Dict]) -> Dict[str, Dict]:
    """
    为所有表格添加序号列
    
    参数:
        tables: 解析得到的表格数据
        
    返回:
        添加了序号列的新表格数据
    """
    
    modified_tables = {}
    
    for level1, level2_dict in tables.items():
        modified_tables[level1] = {}
        
        for level2, table_data in level2_dict.items():
            headers = ["序号"] + table_data["headers"]
            rows = table_data["rows"]
            
            # 为每一行添加序号
            numbered_rows = []
            for i, row in enumerate(rows, 1):
                numbered_rows.append([str(i)] + row)
            
            modified_tables[level1][level2] = {
                "headers": headers,
                "rows": numbered_rows
            }
    
    return modified_tables

def save_tables_with_serial(file_path: str, tables: Dict[str, Dict]) -> None:
    """
    将添加了序号列的表格保存为Markdown文件
    
    参数:
        file_path: 输出文件路径
        tables: 表格数据
    """
    
    with open(file_path, 'w', encoding='utf-8') as f:
        for level1, level2_dict in tables.items():
            # 写入一级标题
            f.write(f"# {level1}\n\n")
            
            for level2, table_data in level2_dict.items():
                # 写入二级标题
                f.write(f"## {level2}\n\n")
                
                # 写入表头
                headers = table_data["headers"]
                f.write("| " + " | ".join(headers) + " |\n")
                
                # 写入表头分隔线
                f.write("| " + " | ".join(["---"] * len(headers)) + " |\n")
                
                # 写入数据行
                rows = table_data["rows"]
                for row in rows:
                    f.write("| " + " | ".join(row) + " |\n")
                
                f.write("\n")

# 测试添加序号列功能
if __name__ == "__main__":
    # 加载之前解析的数据
    # with open("parsed_tables1.json", "r", encoding="utf-8") as f:
    #     tables1 = json.load(f)
    
    # with open("parsed_tables2.json", "r", encoding="utf-8") as f:
    #     tables2 = json.load(f)
    
    # 添加序号列
    OCRtable_with_serial = add_serial_number_to_tables(OCRtables)
    ModifyTable_with_serial = add_serial_number_to_tables(ModifyTables)
    
    print("文件1添加序号列后的表格摘要:")
    display_table_summary(OCRtable_with_serial)
    
    print("\n" + "="*60 + "\n")
    
    print("文件2添加序号列后的表格摘要:")
    display_table_summary(ModifyTable_with_serial)
    


文件1添加序号列后的表格摘要:
表格解析摘要

一级标题: 检字表
  ├─ 二级标题: 检字
  │  ├─ 表头: ['序号', '笔画数', '字']
  │  ├─ 行数: 21
  │  └─ 列数: 3
  ├─ 二级标题: 替换字
  │  ├─ 表头: ['序号', '现代汉语用字', '《法式》用字']
  │  ├─ 行数: 21
  │  └─ 列数: 3

一级标题: 术语解释表
  ├─ 二级标题: 二画
  │  ├─ 表头: ['序号', '术语', '首次出现卷数', '解释']
  │  ├─ 行数: 10
  │  └─ 列数: 4
  ├─ 二级标题: 三画
  │  ├─ 表头: ['序号', '术语', '首次出现卷数', '解释']
  │  ├─ 行数: 45
  │  └─ 列数: 4
  ├─ 二级标题: 四画
  │  ├─ 表头: ['序号', '术语', '首次出现卷数', '解释']
  │  ├─ 行数: 66
  │  └─ 列数: 4
  ├─ 二级标题: 五 画
  │  ├─ 表头: ['序号', '术语', '首次出现卷数', '解释']
  │  ├─ 行数: 53
  │  └─ 列数: 4
  ├─ 二级标题: 六画
  │  ├─ 表头: ['序号', '术语', '首次出现卷数', '解释']
  │  ├─ 行数: 65
  │  └─ 列数: 4
  ├─ 二级标题: 七画
  │  ├─ 表头: ['序号', '术语', '首次出现卷数', '解释']
  │  ├─ 行数: 65
  │  └─ 列数: 4
  ├─ 二级标题: 八画
  │  ├─ 表头: ['序号', '术语', '首次出现卷数', '解释']
  │  ├─ 行数: 87
  │  └─ 列数: 4
  ├─ 二级标题: 九画
  │  ├─ 表头: ['序号', '术语', '首次出现卷数', '解释']
  │  ├─ 行数: 57
  │  └─ 列数: 4
  ├─ 二级标题: 十画
  │  ├─ 表头: ['序号', '术语', '首次出现卷数', '解释']
  │  ├─ 行数: 42
  │  └─ 列数: 4
  ├─ 二级标题: 十一画
  │  ├─ 表头: ['序号', '术语',

In [ ]:
# 保存序列化数据供后续使用
with open("OCRtable.json", "w", encoding="utf-8") as f:
    json.dump(OCRtable_with_serial, f, ensure_ascii=False, indent=2)
    print("OCRtable.json 已保存")

with open("ModifyTable.json", "w", encoding="utf-8") as f:
    json.dump(ModifyTable_with_serial, f, ensure_ascii=False, indent=2)
    print("ModifyTable.json 已保存")

## 步骤3：表格对比分析

In [ ]:
import difflib
from typing import List, Dict, Any

def normalize_row(row: List[str]) -> str:
    """
    将一行数据规范化为一个字符串用于对比
    
    参数:
        row: 一行数据（列表形式）
        
    返回:
        规范化的字符串：第一列 + "：" + 剩余列拼接
    """
    if not row:
        return ""
    
    # 去除每列的多余空格
    cleaned = [col.strip() for col in row]
    
    # 第一列单独处理
    first_col = cleaned[0]
    
    # 剩余列直接拼接
    remaining = "".join(cleaned[1:])
    
    return f"{first_col}：{remaining}"

def compare_tables(file1_tables: Dict[str, Dict], file2_tables: Dict[str, Dict]) -> List[Dict[str, Any]]:
    """
    对比两个文件中的所有表格
    
    参数:
        file1_tables: 文件1的表格数据
        file2_tables: 文件2的表格数据
        
    返回:
        修正清单列表，每个元素是一个修正记录
    """
    
    correction_list = []
    correction_id = 1
    
    # 遍历文件1的所有表格
    for level1, level2_dict1 in file1_tables.items():
        for level2, table_data1 in level2_dict1.items():
            # 查找对应的表格（根据一级标题和二级标题）
            if level1 in file2_tables and level2 in file2_tables[level1]:
                table_data2 = file2_tables[level1][level2]
                
                # 检查表格行数是否一致
                rows1 = table_data1["rows"]
                rows2 = table_data2["rows"]
                
                if len(rows1) != len(rows2):
                    print(f"警告: {level1}:{level2} 行数不一致 ({len(rows1)} vs {len(rows2)})")
                    # 取较小行数进行对比
                    min_rows = min(len(rows1), len(rows2))
                    rows1 = rows1[:min_rows]
                    rows2 = rows2[:min_rows]
                
                # 对比每一行（跳过序号列）
                for i in range(len(rows1)):
                    row1 = rows1[i][1:]  # 跳过序号列
                    row2 = rows2[i][1:]  # 跳过序号列
                    
                    # 规范化行数据
                    row1_str = normalize_row(row1)
                    row2_str = normalize_row(row2)
                    
                    # 如果两行不同，记录修正
                    if row1_str != row2_str:
                        # 分析修改内容
                        modification = analyze_modification(row1_str, row2_str)
                        
                        # 创建修正记录
                        correction_record = {
                            "编号": correction_id,
                            "表来源": f"{level1}：{level2}",
                            "行来源": f"第{i+1}行",
                            "OCR解析结果": row1_str,
                            "人工复核结果": row2_str,
                            "修改内容": modification
                        }
                        
                        correction_list.append(correction_record)
                        correction_id += 1
    
    return correction_list

def analyze_modification(original: str, corrected: str) -> str:
    """
    分析两个字符串之间的修改内容
    
    参数:
        original: 原始字符串
        corrected: 修正后的字符串
        
    返回:
        修改内容描述字符串
    """
    
    # 使用difflib进行对比
    d = difflib.Differ()
    diff = list(d.compare(original, corrected))
    
    # 分析差异
    added_chars = []
    deleted_chars = []
    modified_pairs = []
    
    i = 0
    while i < len(diff):
        if diff[i].startswith('+ '):
            # 添加的字符
            added_chars.append(diff[i][2:])
        elif diff[i].startswith('- '):
            # 删除的字符
            if i + 1 < len(diff) and diff[i+1].startswith('+ '):
                # 修改操作
                modified_pairs.append(f"{diff[i][2:]}→{diff[i+1][2:]}")
                i += 1
            else:
                deleted_chars.append(diff[i][2:])
        i += 1
    
    # 构建修改内容字符串
    modification_parts = []
    
    if added_chars:
        modification_parts.append(f"增：{''.join(added_chars)}")
    
    if deleted_chars:
        modification_parts.append(f"删：{''.join(deleted_chars)}")
    
    if modified_pairs:
        modification_parts.append(f"改：{'，'.join(modified_pairs)}")
    
    return " / ".join(modification_parts) if modification_parts else "无修改"

def display_correction_summary(correction_list: List[Dict[str, Any]]) -> None:
    """显示修正清单摘要"""
    print("=" * 80)
    print("修正清单摘要")
    print("=" * 80)
    
    if not correction_list:
        print("未发现任何修改")
        return
    
    print(f"发现 {len(correction_list)} 处修改:")
    print()
    
    # 按表来源分组统计
    table_stats = {}
    for record in correction_list:
        table_source = record["表来源"]
        if table_source not in table_stats:
            table_stats[table_source] = 0
        table_stats[table_source] += 1
    
    print("按表格分布的修改数量:")
    for table, count in table_stats.items():
        print(f"  {table}: {count} 处修改")
    
    print()
    print("详细修改记录:")
    print("-" * 80)
    
    for record in correction_list:
        print(f"编号: {record['编号']}")
        print(f"表来源: {record['表来源']}")
        print(f"行来源: {record['行来源']}")
        print(f"OCR解析结果: {record['OCR解析结果'][:50]}..." if len(record['OCR解析结果']) > 50 else f"OCR解析结果: {record['OCR解析结果']}")
        print(f"人工复核结果: {record['人工复核结果'][:50]}..." if len(record['人工复核结果']) > 50 else f"人工复核结果: {record['人工复核结果']}")
        print(f"修改内容: {record['修改内容']}")
        print("-" * 40)

# 测试对比功能
if __name__ == "__main__":
    
    # 对比表格
    print("开始对比表格...")
    correction_list = compare_tables(OCRtable_with_serial, ModifyTable_with_serial)
    
    # 显示摘要
    display_correction_summary(correction_list)
    

开始对比表格...
修正清单摘要
发现 153 处修改:

按表格分布的修改数量:
  检字表：检字: 12 处修改
  术语解释表：三画: 3 处修改
  术语解释表：四画: 6 处修改
  术语解释表：五 画: 2 处修改
  术语解释表：六画: 2 处修改
  术语解释表：七画: 6 处修改
  术语解释表：八画: 11 处修改
  术语解释表：九画: 22 处修改
  术语解释表：十画: 12 处修改
  术语解释表：十一画: 12 处修改
  术语解释表：十二画: 21 处修改
  术语解释表：十三画: 14 处修改
  术语解释表：十四画: 9 处修改
  术语解释表：十五画: 7 处修改
  术语解释表：十六画: 5 处修改
  术语解释表：十七画: 4 处修改
  术语解释表：十八画: 1 处修改
  术语解释表：二十一画: 1 处修改
  术语解释表：二十四画: 2 处修改
  术语解释表：二十五画: 1 处修改

详细修改记录:
--------------------------------------------------------------------------------
编号: 1
表来源: 检字表：检字
行来源: 第1行
OCR解析结果: 二画：丁七八八九
人工复核结果: 二画：丁七八入九
修改内容: 改：八→入
----------------------------------------
编号: 2
表来源: 检字表：检字
行来源: 第9行
OCR解析结果: 十画：素栔桩栱枤格荻起破套剔柴铁笏透脊狼鸱鸾流调阁展难
人工复核结果: 十画：素栔桩栱枤格荻起破套剔柴铁笏透脊狼鸱鸳流调？展难
修改内容: 改：鸾→鸳，阁→？
----------------------------------------
编号: 3
表来源: 检字表：检字
行来源: 第10行
OCR解析结果: 十一画：琉梢桯棂梭壶副营黄曹厢捧排虚堂常偷斜彩象旋望廊减混粗断剪兽着窑随隐骑续绰
人工复核结果: 十一画：琉梢桯棂梭壸副营黄曹厢捧排虚堂常偷斜彩象旋望廊减混粗断剪兽着窑随隐骑续绰
修改内容: 改：壶→壸
----------------------------------------
编号: 4
表来源: 检字表：检字
行来源: 第11行
OCR解析结果:

In [17]:
 # 保存修正清单供后续使用
with open("correction_list.json", "w", encoding="utf-8") as f:
    json.dump(correction_list, f, ensure_ascii=False, indent=2)

print(f"\n检字与术语修正清单已保存到 correction_list.json")


检字与术语修正清单已保存到 correction_list.json


## 步骤4：生成修正清单表

In [ ]:
def generate_correction_markdown(correction_list: List[Dict[str, Any]], output_file: str) -> None:
    """
    生成Markdown格式的修正清单表
    
    参数:
        correction_list: 修正记录列表
        output_file: 输出文件路径
    """
    
    with open(output_file, 'w', encoding='utf-8') as f:
        # 写入标题
        f.write("# 检字与术语解释修正清单表\n\n")
        
        # 写入表格说明
        f.write("**说明**: 本表格记录了OCR解析结果与人工复核结果的差异。\n\n")
        
        # 写入表头
        headers = ["编号", "表来源", "行来源", "OCR解析结果", "人工复核结果", "修改内容"]
        f.write("| " + " | ".join(headers) + " |\n")
        f.write("| " + " | ".join(["---"] * len(headers)) + " |\n")
        
        # 写入数据行
        for record in correction_list:
            # 处理可能包含换行符或过长文本的情况
            ocr_result = record["OCR解析结果"].replace("\n", " ").replace("|", "\\|")
            manual_result = record["人工复核结果"].replace("\n", " ").replace("|", "\\|")
            modification = record["修改内容"].replace("\n", " ").replace("|", "\\|")
            
            # 如果文本过长，适当截断
            if len(ocr_result) > 50:
                ocr_result = ocr_result[:47] + "..."
            if len(manual_result) > 50:
                manual_result = manual_result[:47] + "..."
            
            row = [
                str(record["编号"]),
                record["表来源"],
                record["行来源"],
                ocr_result,
                manual_result,
                modification
            ]
            
            f.write("| " + " | ".join(row) + " |\n")
        
        # 添加统计信息
        f.write(f"\n**总计**: {len(correction_list)} 处修改\n")
        
        # 按表来源统计
        table_stats = {}
        for record in correction_list:
            table_source = record["表来源"]
            if table_source not in table_stats:
                table_stats[table_source] = 0
            table_stats[table_source] += 1
        
        f.write("\n**按表格分布**:\n")
        for table, count in table_stats.items():
            f.write(f"- {table}: {count} 处修改\n")

def generate_detailed_report(correction_list: List[Dict[str, Any]], output_file: str) -> None:
    """
    生成详细的HTML格式报告，包含差异高亮显示
    
    参数:
        correction_list: 修正记录列表
        output_file: 输出文件路径
    """
    
    html_content = """
<!DOCTYPE html>
<html lang="zh-CN">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>检字与术语解释修正清单表</title>
    <style>
        body {
            font-family: 'Microsoft YaHei', Arial, sans-serif;
            line-height: 1.6;
            margin: 0;
            padding: 20px;
            background-color: #f5f5f5;
        }
        .container {
            max-width: 1200px;
            margin: 0 auto;
            background-color: white;
            padding: 30px;
            border-radius: 10px;
            box-shadow: 0 2px 10px rgba(0,0,0,0.1);
        }
        h1 {
            color: #2c3e50;
            border-bottom: 3px solid #3498db;
            padding-bottom: 10px;
            margin-bottom: 30px;
        }
        .summary {
            background-color: #f8f9fa;
            padding: 15px;
            border-radius: 5px;
            margin-bottom: 20px;
            border-left: 4px solid #3498db;
        }
        table {
            width: 100%;
            border-collapse: collapse;
            margin-bottom: 30px;
        }
        th {
            background-color: #3498db;
            color: white;
            padding: 12px;
            text-align: left;
            position: sticky;
            top: 0;
        }
        td {
            padding: 12px;
            border-bottom: 1px solid #ddd;
        }
        tr:nth-child(even) {
            background-color: #f8f9fa;
        }
        tr:hover {
            background-color: #e8f4fc;
        }
        .diff-added {
            background-color: #d4edda;
            color: #155724;
            padding: 2px 4px;
            border-radius: 3px;
        }
        .diff-removed {
            background-color: #f8d7da;
            color: #721c24;
            padding: 2px 4px;
            border-radius: 3px;
            text-decoration: line-through;
        }
        .diff-modified {
            background-color: #fff3cd;
            color: #856404;
            padding: 2px 4px;
            border-radius: 3px;
        }
        .stat-card {
            display: inline-block;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            padding: 15px;
            border-radius: 8px;
            margin: 10px;
            min-width: 200px;
        }
        .stat-card h3 {
            margin: 0 0 10px 0;
            font-size: 16px;
        }
        .stat-card p {
            margin: 0;
            font-size: 24px;
            font-weight: bold;
        }
        .table-stats {
            display: flex;
            flex-wrap: wrap;
            justify-content: center;
            margin-bottom: 30px;
        }
    </style>
</head>
<body>
    <div class="container">
        <h1>📋 检字与术语解释修正清单表</h1>
        
        <div class="summary">
            <h2>📊 总体统计</h2>
            <div class="table-stats">
                <div class="stat-card">
                    <h3>总修改数量</h3>
                    <p id="total-corrections">0</p>
                </div>
            </div>
        </div>
        
        <h2>🔍 详细修改记录</h2>
        <table id="correction-table">
            <thead>
                <tr>
                    <th>编号</th>
                    <th>表来源</th>
                    <th>行来源</th>
                    <th>OCR解析结果</th>
                    <th>人工复核结果</th>
                    <th>修改内容</th>
                </tr>
            </thead>
            <tbody>
"""
    
    # 按表来源分组统计
    table_stats = {}
    for record in correction_list:
        table_source = record["表来源"]
        if table_source not in table_stats:
            table_stats[table_source] = 0
        table_stats[table_source] += 1
    
    # 添加统计卡片
    for table, count in table_stats.items():
        html_content += f"""
                <div class="stat-card">
                    <h3>{table}</h3>
                    <p>{count} 处修改</p>
                </div>"""
    
    # 添加表格行
    for record in correction_list:
        # 高亮显示差异
        ocr_result = highlight_differences(record["OCR解析结果"], record["人工复核结果"], is_ocr=True)
        manual_result = highlight_differences(record["OCR解析结果"], record["人工复核结果"], is_ocr=False)
        
        html_content += f"""
                <tr>
                    <td>{record["编号"]}</td>
                    <td>{record["表来源"]}</td>
                    <td>{record["行来源"]}</td>
                    <td>{ocr_result}</td>
                    <td>{manual_result}</td>
                    <td><span class="diff-modified">{record["修改内容"]}</span></td>
                </tr>"""
    
    html_content += """
            </tbody>
        </table>
        
        <script>
            // 更新总修改数量
            document.getElementById('total-corrections').textContent = 
                document.querySelectorAll('#correction-table tbody tr').length;
        </script>
    </div>
</body>
</html>"""
    
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(html_content)

def highlight_differences(original: str, corrected: str, is_ocr: bool = True) -> str:
    """
    高亮显示两个字符串之间的差异
    
    参数:
        original: 原始字符串
        corrected: 修正后的字符串
        is_ocr: 是否为OCR结果（True显示删除的内容，False显示添加的内容）
        
    返回:
        带HTML标记的字符串
    """
    if original == corrected:
        return original
    
    # 简单的差异高亮（实际应用中可以使用更复杂的算法）
    import difflib
    
    d = difflib.Differ()
    diff = list(d.compare(original, corrected))
    
    result = ""
    i = 0
    
    while i < len(diff):
        if is_ocr:
            # 对于OCR结果，显示被删除的部分
            if diff[i].startswith('- '):
                result += f'<span class="diff-removed">{diff[i][2:]}</span>'
                i += 1
            elif diff[i].startswith('+ '):
                i += 1
            elif diff[i].startswith('  '):
                result += diff[i][2:]
                i += 1
        else:
            # 对于人工结果，显示被添加的部分
            if diff[i].startswith('+ '):
                result += f'<span class="diff-added">{diff[i][2:]}</span>'
                i += 1
            elif diff[i].startswith('- '):
                i += 1
            elif diff[i].startswith('  '):
                result += diff[i][2:]
                i += 1
    
    return result if result else ""

# 测试生成修正清单
if __name__ == "__main__":
    
    # 生成Markdown格式的修正清单
    generate_correction_markdown(correction_list, "correction_checklist.md")
    print("输出correction_checklist.md - Markdown格式的修正清单表")
    # 显示生成的Markdown内容预览
    print("修正清单表预览（前3行）:")
    print("="*80)
    with open("correction_checklist.md", "r", encoding="utf-8") as f:
        lines = f.readlines()
        for i, line in enumerate(lines):
            if i < 10:  # 显示前10行
                print(line.rstrip())
    
    # 生成HTML详细报告
    generate_detailed_report(correction_list, "correction_report.html")
    print("输出correction_report.html - HTML格式的详细报告（带差异高亮）")
    

    print("\n完成！")

输出correction_checklist.md - Markdown格式的修正清单表
# 检字与术语解释修正清单表

**说明**: 本表格记录了OCR解析结果与人工复核结果的差异。

| 编号 | 表来源 | 行来源 | OCR解析结果 | 人工复核结果 | 修改内容 |
| --- | --- | --- | --- | --- | --- |
| 1 | 检字表：检字 | 第1行 | 二画：丁七八八九 | 二画：丁七八入九 | 改：八→入 |
| 2 | 检字表：检字 | 第9行 | 十画：素栔桩栱枤格荻起破套剔柴铁笏透脊狼鸱鸾流调阁展难 | 十画：素栔桩栱枤格荻起破套剔柴铁笏透脊狼鸱鸳流调？展难 | 改：鸾→鸳，阁→？ |
| 3 | 检字表：检字 | 第10行 | 十一画：琉梢桯棂梭壶副营黄曹厢捧排虚堂常偷斜彩象旋望廊减混粗断剪兽着窑随隐骑续绰 | 十一画：琉梢桯棂梭壸副营黄曹厢捧排虚堂常偷斜彩象旋望廊减混粗断剪兽着窑随隐骑续绰 | 改：壶→壸 |
| 4 | 检字表：检字 | 第11行 | 十二画：替琴棵榁𬃊棚散惹葱趄欹厦雁颊搭插辋赑铺锭鹅筒牌竣敦阑普隔缘 | 十二画：替琴棵㭼𬃊棚散惹葱趄欹厦雁颊搭插辋赑铺？鹅筒牌竣敦阑普隔缘 | 改：榁→㭼，锭→？ |
输出correction_report.html - HTML格式的详细报告（带差异高亮）
修正清单表预览（前3行）:

完成！
